In [1]:
#2d implementation progression

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time

torch.manual_seed(123)
np.random.seed(123)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
from pinn_shared import (
    set_seed, pinn_architecture, sample_points, generate_noisy_data,
    compute_loss, train_forward, compute_loss_inverse, train_inverse,
    fd_solver, cn_nls_baseline,
)

In [3]:
forward_config = {
    "N_f": 10000,
    "N_bc": 200,
    "N_ic": 200,
    "adam_lr": 1e-3,
    "adam_iters": 2000,
    "lbfgs_iters": 500,
    "hidden_size": 20,
    "n_layers": 4,
}

In [6]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_forward(trial):
    config = {
        "hidden_size":  trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
        "n_layers":     trial.suggest_categorical("n_layers", [3, 4, 5]),
        "activation":   trial.suggest_categorical("activation", ["tanh", "sin"]),
        "lambda_pde":   1,
        "lambda_bc":    trial.suggest_float("lambda_bc",  0.001, 1000.0, log=True),
        "lambda_ic":    trial.suggest_float("lambda_ic",  0.001, 1000.0, log=True),
        "N_f":          trial.suggest_categorical("N_f", [5000, 10000, 20000]),
        "N_bc":         trial.suggest_categorical("N_bc", [100, 200, 400]),
        "N_ic":         trial.suggest_categorical("N_ic", [100, 200, 400]),
        "adam_lr":      trial.suggest_float("adam_lr", 1e-4, 1e-2, log=True),
        "adam_iters":   1500,
        "lbfgs_iters":  500,
    }

    results = train_forward(config, print_training = False, trial = trial)
    return results["rel_l2"]

In [7]:
import os
if os.path.exists("optuna_forward.db"):
    os.remove("optuna_forward.db")


In [8]:
import joblib
import optuna.visualization as vis

study = optuna.create_study(
    direction="minimize",
    study_name="forward_pinn_hpo",
    storage="sqlite:///optuna_forward.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=800, interval_steps=200),
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective_forward, 
    n_trials=100,
    show_progress_bar=True
)

joblib.dump(study, 'forward_hpo_study.pkl')

print("\nBest trial:")
print(f"  rel_l2: {study.best_trial.value:.4e}")
print(f"  params: {study.best_trial.params}")

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

  0%|          | 0/100 [00:00<?, ?it/s]


PINN total training time: 12.60s
Rel L2 error: 9.9854e-01

PINN total training time: 216.99s
Rel L2 error: 5.3193e-03

PINN total training time: 95.95s
Rel L2 error: 7.1610e-04

PINN total training time: 185.16s
Rel L2 error: 6.7466e-01

PINN total training time: 77.00s
Rel L2 error: 9.0685e-04

PINN total training time: 152.45s
Rel L2 error: 1.1857e-03

PINN total training time: 84.90s
Rel L2 error: 2.2685e-02

PINN total training time: 187.60s
Rel L2 error: 9.0617e-04

PINN total training time: 118.82s
Rel L2 error: 1.6348e-02

PINN total training time: 75.17s
Rel L2 error: 1.5158e-03

PINN total training time: 50.22s
Rel L2 error: 2.4438e-03

PINN total training time: 41.07s
Rel L2 error: 2.4016e-03

PINN total training time: 226.28s
Rel L2 error: 7.1650e-03

PINN total training time: 25.57s
Rel L2 error: 6.1457e-03

PINN total training time: 32.60s
Rel L2 error: 1.9860e-03

PINN total training time: 362.07s
Rel L2 error: 3.3820e-02

PINN total training time: 55.56s
Rel L2 error: 3

In [9]:
import optuna

study = optuna.load_study(
    study_name="forward_pinn_hpo",
    storage="sqlite:///optuna_forward.db"
)

print(f"\nBest trial:")
print(f"  Error: {study.best_value:.6e}")
print(f"  Config: {study.best_params}")

# See all top configs
print(f"\nTop 5 trials:")
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('inf'))
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"{i}. Error: {trial.value:.6e}")
    print(f"   Params: {trial.params}\n")

# Stats
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Summary:")
print(f"  Completed: {len(completed)}")
print(f"  Pruned: {len(pruned)}")
print(f"  Pruning saved: {len(pruned)/(len(completed)+len(pruned))*100:.1f}% of trials")


Best trial:
  Error: 1.692019e-04
  Config: {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 16.64287157253686, 'lambda_ic': 1.3287678633804623, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.005104288463439231}

Top 5 trials:
1. Error: 1.692019e-04
   Params: {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 16.64287157253686, 'lambda_ic': 1.3287678633804623, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.005104288463439231}

2. Error: 1.781247e-04
   Params: {'hidden_size': 128, 'n_layers': 5, 'activation': 'tanh', 'lambda_bc': 18.312606180932846, 'lambda_ic': 1.2172733015396069, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.004818229052622651}

3. Error: 1.845841e-04
   Params: {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 7.342508201207911, 'lambda_ic': 1.7281707586350386, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.00753569599541483}

4. Error: 2.000675e-04
   Params: {'hidden_size': 1

In [ ]:
top_configs = [
    {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 16.64287157253686, 'lambda_ic': 1.3287678633804623,
     'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.005104288463439231, 'adam_iters': 3000, 'lbfgs_iters': 1500},

    {'hidden_size': 128, 'n_layers': 5, 'activation': 'tanh', 'lambda_bc': 18.312606180932846, 'lambda_ic': 1.2172733015396069,
     'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.004818229052622651, 'adam_iters': 3000, 'lbfgs_iters': 1500},

    {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 7.342508201207911, 'lambda_ic': 1.7281707586350386,
     'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.00753569599541483, 'adam_iters': 3000, 'lbfgs_iters': 1500},
]

# Held-out test grid -- same construction as the final PINN-vs-CN comparison
# cell below -- built here so every seed's model can be checked against it,
# not just whichever single run ends up being visualized.
x_test = torch.linspace(0, 1, 1000)
t_test = torch.linspace(0, 1, 1000)
X_test, T_test = torch.meshgrid(x_test, t_test, indexing='ij')
x_test_flat = X_test.reshape(-1, 1)
t_test_flat = T_test.reshape(-1, 1)
u_exact_test = torch.sin(torch.pi * x_test_flat) * torch.exp(-0.4 * (torch.pi**2) * t_test_flat)

# Train every (config, seed) pair and keep every result -- not just the
# single luckiest one -- so config selection and final reporting can both
# be based on average behavior instead of a cherry-picked best run.
all_config_results = []

for i, config in enumerate(top_configs, 1):
    config_results = []
    for seed in range(5):
        set_seed(seed)
        start = time.time()
        results = train_forward(config, print_training=False)
        results['elapsed'] = time.time() - start
        config_results.append(results)

    errors = [r['rel_l2'] for r in config_results]
    times = [r['elapsed'] for r in config_results]
    print(f"\nConfig {i}:")
    print(f"  Error (validation grid): {np.mean(errors):.6e} ± {np.std(errors):.6e}")
    print(f"  Time:  {np.mean(times):.2f} ± {np.std(times):.2f} sec")

    all_config_results.append(config_results)

# Pick the winning config by its MEAN validation error across 5 seeds, not
# by any single individual run's error -- so config selection isn't biased
# toward whichever run happened to get an easy random draw.
config_mean_errors = [np.mean([r['rel_l2'] for r in cr]) for cr in all_config_results]
best_config_idx = int(np.argmin(config_mean_errors))
winning_results = all_config_results[best_config_idx]

# Evaluate every seed of the winning config on the held-out TEST grid
# (never used during HPO or config selection) -- this mean/std, not any
# single run's number, is what should be quoted as "PINN accuracy."
test_errors = []
for r in winning_results:
    with torch.no_grad():
        u_pred_test = r["model"](x_test_flat, t_test_flat)
    test_errors.append((torch.norm(u_pred_test - u_exact_test) / torch.norm(u_exact_test)).item())

pinn_test_rel_l2_mean = float(np.mean(test_errors))
pinn_test_rel_l2_std = float(np.std(test_errors))
pinn_train_time_mean = float(np.mean([r["elapsed"] for r in winning_results]))
pinn_train_time_std = float(np.std([r["elapsed"] for r in winning_results]))

print(f"\nWinning config: Config {best_config_idx + 1}")
print(f"  Held-out test rel L2 error: {pinn_test_rel_l2_mean:.6e} ± {pinn_test_rel_l2_std:.6e}")
print(f"  Training time:              {pinn_train_time_mean:.2f} ± {pinn_train_time_std:.2f} sec")

# Use whichever seed's validation error is closest to the winning config's
# mean as a single representative model for the plots below -- illustrative
# only. The mean ± std above are the numbers that should be quoted as PINN
# performance, not this one run's individual numbers.
representative_idx = int(np.argmin([abs(r['rel_l2'] - config_mean_errors[best_config_idx]) for r in winning_results]))
best_result = winning_results[representative_idx]

model = best_result["model"]
alpha = 0.4
pinn_train_time = best_result["train_time"]

In [ ]:
# Baseline (no-tuning) forward PINN: the literature-informed hyperparameters
# from heat_pinn_basic.ipynb (forward_config above), run through the same
# multi-seed / held-out-test-grid methodology as the tuned configs, so it's
# directly comparable to both the tuned PINN and CN -- this is the "PINN
# performance you'd get without paying for Optuna tuning" number.
baseline_forward_results = []
for seed in range(5):
    set_seed(seed)
    start = time.time()
    results = train_forward(forward_config, print_training=False)
    results['elapsed'] = time.time() - start
    baseline_forward_results.append(results)

baseline_val_errors = [r['rel_l2'] for r in baseline_forward_results]
baseline_val_times = [r['elapsed'] for r in baseline_forward_results]
print(f"\nBaseline (no tuning):")
print(f"  Error (validation grid): {np.mean(baseline_val_errors):.6e} ± {np.std(baseline_val_errors):.6e}")
print(f"  Time:  {np.mean(baseline_val_times):.2f} ± {np.std(baseline_val_times):.2f} sec")

# Evaluate every seed on the same held-out TEST grid used for the tuned
# config above, so baseline vs. tuned vs. CN are all reported on identical
# points.
baseline_test_errors = []
for r in baseline_forward_results:
    with torch.no_grad():
        u_pred_test = r["model"](x_test_flat, t_test_flat)
    baseline_test_errors.append((torch.norm(u_pred_test - u_exact_test) / torch.norm(u_exact_test)).item())

baseline_test_rel_l2_mean = float(np.mean(baseline_test_errors))
baseline_test_rel_l2_std = float(np.std(baseline_test_errors))
baseline_train_time_mean = float(np.mean(baseline_val_times))
baseline_train_time_std = float(np.std(baseline_val_times))

print(f"  Held-out test rel L2 error: {baseline_test_rel_l2_mean:.6e} ± {baseline_test_rel_l2_std:.6e}")
print(f"  Training time:              {baseline_train_time_mean:.2f} ± {baseline_train_time_std:.2f} sec")

In [ ]:
# Reuse the held-out test grid built above (x_test/t_test/X_test/T_test/...)
# instead of rebuilding an identical one from scratch -- same points, so
# there's no reason to maintain two copies of the same grid construction.
x, t = x_test, t_test
X, T = X_test, T_test
x_flat, t_flat = x_test_flat, t_test_flat

pinn_infer_start = time.time()
with torch.no_grad():
    u_pred = model(x_flat, t_flat)
pinn_infer_time = time.time() - pinn_infer_start

u_exact = u_exact_test

error = torch.norm(u_pred - u_exact) / torch.norm(u_exact)
linf_error_pinn = torch.max(torch.abs(u_pred - u_exact)).item()

print(f"PINN inference time : {pinn_infer_time:.4f}s")
print(f"PINN rel L2 error   : {error.item():.4e}")
print(f"PINN L-inf error    : {linf_error_pinn:.4e}")

u_pred_grid = u_pred.reshape(1000, 1000)
u_exact_grid = u_exact.reshape(1000, 1000)
u_abs_error = torch.abs(u_pred - u_exact).reshape(1000, 1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1 - exact solution
im1 = axes[0].pcolormesh(T.numpy(), X.numpy(), u_exact_grid.numpy(), cmap='hot')
axes[0].set_title('Exact Solution')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x')
plt.colorbar(im1, ax=axes[0])

# Plot 2 - PINN prediction
im2 = axes[1].pcolormesh(T.numpy(), X.numpy(), u_pred_grid.numpy(), cmap='hot')
axes[1].set_title('PINN Prediction')
axes[1].set_xlabel('t')
axes[1].set_ylabel('x')
plt.colorbar(im2, ax=axes[1])

# Plot 3 - absolute error
im3 = axes[2].pcolormesh(T.numpy(), X.numpy(), u_abs_error.numpy(), cmap='hot')
axes[2].set_title('Absolute Error')
axes[2].set_xlabel('t')
axes[2].set_ylabel('x')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
conv_errors = []
N_values = [50, 100, 200, 400, 800]

for nx in N_values:
    x_c, t_c, u_c, conv_error = fd_solver(nx, nx * 2)
    conv_errors.append(conv_error)

In [ ]:
plt.figure()
plt.loglog(N_values, conv_errors, 'bo-', label='CN Error')
plt.xlabel('N_x')
plt.ylabel('Relative L2 Error')
plt.title('FD Convergence')
plt.grid(True)
plt.show()

In [ ]:
fd_start = time.time()
x_fd, t_fd, u_fd, _ = fd_solver(1000, 1000, compute_error=False)
fd_time = time.time() - fd_start

X_fd, T_fd = np.meshgrid(x_fd, t_fd, indexing='ij')
u_exact_fd = np.sin(np.pi * X_fd) * np.exp(-0.4 * (np.pi**2) * T_fd)
error_fd = np.linalg.norm(u_fd - u_exact_fd) / np.linalg.norm(u_exact_fd)
linf_error_fd = np.max(np.abs(u_fd - u_exact_fd))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Exact
im1 = axes[0].pcolormesh(T.numpy(), X.numpy(), u_exact_grid.numpy(), cmap='hot')
axes[0].set_title('Exact')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x')
plt.colorbar(im1, ax=axes[0])

# PINN prediction
im2 = axes[1].pcolormesh(T.numpy(), X.numpy(), u_pred_grid.numpy(), cmap='hot')
axes[1].set_title('PINN')
axes[1].set_xlabel('t')
plt.colorbar(im2, ax=axes[1])

# FD prediction
im3 = axes[2].pcolormesh(T.numpy(), X.numpy(), u_fd, cmap='hot')
axes[2].set_title('FD (CN)')
axes[2].set_xlabel('t')
plt.colorbar(im3, ax=axes[2])

# PINN error
im4 = axes[3].pcolormesh(T.numpy(), X.numpy(), u_abs_error.numpy(), cmap='hot')
axes[3].set_title('PINN Error')
axes[3].set_xlabel('t')
plt.colorbar(im4, ax=axes[3])

# FD error
im5 = axes[4].pcolormesh(T.numpy(), X.numpy(), np.abs(u_fd - u_exact_fd), cmap='hot')
axes[4].set_title('FD Error')
axes[4].set_xlabel('t')
plt.colorbar(im5, ax=axes[4])

plt.tight_layout()
plt.show()

# Note: the "PINN" column below is a single representative run (the seed
# closest to the winning config's mean), shown for illustration alongside
# the plots above. The mean ± std rows are across the winning config's 5
# seeds, evaluated on the held-out test grid -- those are the numbers that
# should be quoted as PINN performance, not the single-run row.
print(f"{'':25} {'PINN':>12} {'FD (CN)':>12}")
print(f"{'-'*50}")
print(f"{'Training time (s)':25} {pinn_train_time:>12.2f} {'N/A':>12}")
print(f"{'  mean ± std (5 seeds)':25} {f'{pinn_train_time_mean:.2f}±{pinn_train_time_std:.2f}':>12} {'N/A':>12}")
print(f"{'Inference time (s)':25} {pinn_infer_time:>12.4f} {fd_time:>12.4f}")
print(f"{'Rel L2 error':25} {error.item():>12.4e} {error_fd:>12.4e}")
print(f"{'  mean ± std (5 seeds)':25} {f'{pinn_test_rel_l2_mean:.4e}±{pinn_test_rel_l2_std:.4e}':>12} {'N/A':>12}")
print(f"{'L-inf error':25} {linf_error_pinn:>12.4e} {linf_error_fd:>12.4e}")

In [ ]:
inverse_config = {
    "activation":   "sin",
    "hidden_size":  128,
    "n_layers":     4,
    "lambda_pde":   1.0,
    "lambda_bc":    91.5,
    "lambda_ic":    3.13,
    "lambda_data":  0.93,
    "N_f":          10000,
    "N_bc":         200,
    "N_ic":         800,
    "N_obs":        50,
    "noise_std":    0.05,
    "alpha_init":   0.1,
    "true_alpha":   0.4,
    "adam_lr":      0.005,
    "adam_iters":   5000,
    "lbfgs_iters":  500,
    "n_alpha_candidates": 200,
}

In [ ]:
set_seed(0)
x_obs, t_obs, u_obs = generate_noisy_data(inverse_config)
results = train_inverse(inverse_config, x_obs, t_obs, u_obs)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(results["history_total"], label='Total')
axes[0].semilogy(results["history_pde"], label='PDE')
axes[0].semilogy(results["history_bc"], label='BC')
axes[0].semilogy(results["history_ic"], label='IC')
axes[0].semilogy(results["history_data"], label='Data')
axes[0].axvline(x=results["adam_iters"], color='black', linestyle='--', label='Adam → L-BFGS')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Inverse PINN Training Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(results["history_alpha"], label='Recovered alpha')
axes[1].axhline(y=inverse_config["true_alpha"], color='r', linestyle='--', label='True alpha')
axes[1].axvline(x=results["adam_iters"], color='black', linestyle='--', label='Adam → L-BFGS')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Alpha')
axes[1].set_title('Alpha Convergence')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Baseline (no-tuning) inverse PINN: the exact hyperparameters originally
# used in heat_pinn_basic.ipynb's inverse cell (hardcoded tanh activation,
# all loss terms unweighted), the same way forward_config above captures
# the original forward hyperparameters.
baseline_inverse_config = {
    "activation":   "tanh",
    "hidden_size":  20,
    "n_layers":     4,
    "lambda_pde":   1.0,
    "lambda_bc":    1.0,
    "lambda_ic":    1.0,
    "lambda_data":  1.0,
    "N_f":          10000,
    "N_bc":         200,
    "N_ic":         200,
    "N_obs":        10,
    "noise_std":    0.05,
    "alpha_init":   0.1,
    "true_alpha":   0.4,
    "adam_lr":      1e-3,
    "adam_iters":   5000,
    "lbfgs_iters":  500,
}

In [ ]:
# Generate the baseline's own observations once, kept fixed across all 5
# seeds below -- same reasoning as the shared x_obs_inv/t_obs_inv/u_obs_inv
# used for Optuna trials further down: comparing seeds should only reflect
# training stochasticity, not a fresh random observation draw each time.
set_seed(0)
x_obs_baseline, t_obs_baseline, u_obs_baseline = generate_noisy_data(baseline_inverse_config)

baseline_inverse_results = []
for seed in range(5):
    set_seed(seed)
    results = train_inverse(baseline_inverse_config, x_obs_baseline, t_obs_baseline, u_obs_baseline, print_training=False)
    baseline_inverse_results.append(results)

baseline_alpha_errors = [r["alpha_error"] for r in baseline_inverse_results]
baseline_inverse_times = [r["train_time"] for r in baseline_inverse_results]

print(f"\nBaseline (no tuning) inverse PINN:")
print(f"  Alpha error: {np.mean(baseline_alpha_errors):.6e} ± {np.std(baseline_alpha_errors):.6e}")
print(f"  Time: {np.mean(baseline_inverse_times):.2f} ± {np.std(baseline_inverse_times):.2f} sec")

In [ ]:
# Generate the inverse-problem observations once, shared across every Optuna
# trial, so trials are compared on the identical noisy dataset rather than
# each drawing its own random noise realization.
set_seed(0)
x_obs_inv, t_obs_inv, u_obs_inv = generate_noisy_data({
    "N_obs": 50,
    "noise_std": 0.05,
    "true_alpha": 0.4,
})

def objective_inverse(trial):
    config = {
        "hidden_size":  trial.suggest_categorical("hidden_size", [32, 64, 128]),
        "n_layers":     trial.suggest_categorical("n_layers", [3, 4, 5]),
        "activation":   "sin",
        "lambda_pde":   1.0,
        "lambda_bc":    trial.suggest_float("lambda_bc",  0.001, 1000.0, log=True),
        "lambda_ic":    trial.suggest_float("lambda_ic",  0.001, 1000.0, log=True),
        "lambda_data":  trial.suggest_float("lambda_data",  0.001, 1000.0, log=True),
        "N_f":          10000,
        "N_bc":         trial.suggest_categorical("N_bc", [200, 400, 800]),
        "N_ic":         trial.suggest_categorical("N_ic", [200, 400, 800]),
        "N_obs":        50,
        "adam_lr":      trial.suggest_float("adam_lr", 1e-4, 1e-2, log=True),
        "adam_iters":   1500,
        "lbfgs_iters":  500,
        "noise_std":    0.05,
        "alpha_init":   0.1,
        "true_alpha":   0.4,
    }

    results = train_inverse(config, x_obs_inv, t_obs_inv, u_obs_inv, print_training=False, trial=trial)
    return abs(results["alpha"] - config["true_alpha"])

In [ ]:
import os
if os.path.exists("optuna_inverse.db"):
    os.remove("optuna_inverse.db")


In [ ]:
import joblib
import optuna.visualization as vis

study = optuna.create_study(
    direction="minimize",
    study_name="inverse_pinn_hpo",
    storage="sqlite:///optuna_inverse.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=800, interval_steps=200, n_min_trials=8),
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective_inverse, 
    n_trials=100,
    show_progress_bar=True
)

joblib.dump(study, 'inverse_hpo_study.pkl')

print("\nBest trial:")
print(f"  alpha_error: {study.best_trial.value:.4e}")
print(f"  params: {study.best_trial.params}")

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

In [ ]:
study = optuna.load_study(
    study_name="inverse_pinn_hpo",
    storage="sqlite:///optuna_inverse.db"
)

print(f"\nBest trial:")
print(f"  Alpha error: {study.best_value:.6e}")
print(f"  Config: {study.best_params}")

# See all top configs
print(f"\nTop 5 trials:")
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('inf'))
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"{i}. Alpha error: {trial.value:.6e}")
    print(f"   Params: {trial.params}\n")

# Stats
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Summary:")
print(f"  Completed: {len(completed)}")
print(f"  Pruned: {len(pruned)}")
print(f"  Pruning saved: {len(pruned)/(len(completed)+len(pruned))*100:.1f}% of trials")


In [ ]:
best_alpha, mse_history, alpha_candidates, cn_nls_time = cn_nls_baseline(
    inverse_config, results["x_obs"], results["t_obs"], results["u_obs"])

print(f"CN-NLS recovered alpha: {best_alpha:.6f} | True alpha: {inverse_config['true_alpha']} | Error: {abs(best_alpha - inverse_config['true_alpha']):.6f}")

plt.figure(figsize=(8, 4))
plt.plot(alpha_candidates, mse_history, label='MSE')
plt.axvline(x=best_alpha, color='b', linestyle='--', label=f'Best alpha ({best_alpha:.4f})')
plt.axvline(x=inverse_config["true_alpha"], color='r', linestyle='--', label=f'True alpha ({inverse_config["true_alpha"]})')
plt.xlabel('Alpha candidate')
plt.ylabel('MSE')
plt.title('CN-NLS Objective Landscape')
plt.legend()
plt.grid(True)
plt.show()